# 🖼️ Notebook 4: Image Preprocessing
## Vision Transformer Preprocessing Pipeline — Album Cover Art

---

> **Notebook 4 of 9** | Prerequisites: Notebook 1 complete, `subset_50.csv` exists, image files present.

---

### What This Notebook Builds

```
Raw JPEG/PNG image
    → Load as RGB array  (H, W, 3)
    → Resize to 224×224  (224, 224, 3)
    → Normalize          (3, 224, 224) float32 in [−2, 2]
    → Extract patches    (196, 768)  ← 14×14 grid of 16×16×3 patches
    → Save .npy
```

**Output per song:** `(196, 768)` — 196 image patches, each with 768 values  
(a 16×16 RGB patch flattened: 16×16×3 = 768)

**Why 224×224?** Standard ViT input size. Divisible evenly by patch size 16 → 14×14 = 196 patches.

---
## 📐 Mathematical Background

### 4.1 — Why ViT Uses Patches (not pixels)

A 224×224 RGB image has 224×224×3 = 150,528 values.  
If each pixel were a separate token, attention would require O(150,528²) operations — impossible.

Instead, ViT divides the image into non-overlapping patches of size (P × P):  
$$N_{patches} = \left(\frac{H}{P}\right) \times \left(\frac{W}{P}\right) = \frac{224}{16} \times \frac{224}{16} = 14 \times 14 = 196$$

Each patch is flattened: $P^2 \times C = 16^2 \times 3 = 768$ values → projected to `embed_dim=128`.

Attention complexity: O(196²) = 38,416 — very tractable.

### 4.2 — ImageNet Normalization

Standard RGB normalization (ImageNet statistics):

$$x_{norm}[c] = \frac{x[c] / 255 - \mu_c}{\sigma_c}$$

Where:
- `μ = [0.485, 0.456, 0.406]` (R, G, B channel means)
- `σ = [0.229, 0.224, 0.225]` (R, G, B channel stds)

**Why these specific values?** They were computed over 1.2M ImageNet images.  
Since we train from scratch (no ImageNet pretraining), we could also use dataset-specific stats.  
For simplicity and compatibility, we use ImageNet stats — standard practice.

### 4.3 — CLS Token Intuition

After patch embedding, a learnable `[CLS]` token is prepended:  
Input: `[[CLS], patch_1, patch_2, ..., patch_196]` → sequence of 197 tokens  

The `[CLS]` token aggregates information from all patches through attention.  
After the transformer, `output[0]` (the [CLS] position) serves as the **image embedding**.

Alternatively, mean pooling over patch outputs also works — we implement both options in Notebook 5.

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 4.1 — IMPORTS AND CONFIG
# ─────────────────────────────────────────────────────────────

import sys, subprocess
try:
    from PIL import Image
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'Pillow', '-q'])
    from PIL import Image

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from pathlib import Path
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────
BASE_DIR    = Path('music_recommender')
IMAGE_DIR   = BASE_DIR / 'raw' / 'images'
SUBSET_CSV  = BASE_DIR / 'subsets' / 'subset_50.csv'
PROC_IMAGE  = BASE_DIR / 'processed' / 'images'
FIGURES_DIR = BASE_DIR / 'figures' / 'images'

PROC_IMAGE.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ── ViT Hyperparameters ───────────────────────────────────────
IMG_SIZE   = 224   # Resize all images to 224×224 px
PATCH_SIZE = 16    # Each patch is 16×16 px
N_CHANNELS = 3     # RGB

N_PATCHES_PER_DIM = IMG_SIZE // PATCH_SIZE          # = 14
N_PATCHES         = N_PATCHES_PER_DIM ** 2           # = 196
PATCH_DIM         = PATCH_SIZE * PATCH_SIZE * N_CHANNELS  # = 768

# ImageNet normalization constants
IMG_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)  # per channel
IMG_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

print("=" * 50)
print("IMAGE PREPROCESSING CONFIG")
print("=" * 50)
print(f"Input size         : {IMG_SIZE}×{IMG_SIZE}×{N_CHANNELS}")
print(f"Patch size         : {PATCH_SIZE}×{PATCH_SIZE}")
print(f"Patches per dim    : {N_PATCHES_PER_DIM} ({IMG_SIZE}/{PATCH_SIZE})")
print(f"Total patches      : {N_PATCHES}  = {N_PATCHES_PER_DIM}²")
print(f"Patch dimension    : {PATCH_DIM}  = {PATCH_SIZE}²×{N_CHANNELS}")
print(f"Output per song    : ({N_PATCHES}, {PATCH_DIM})")
print(f"Memory/song        : {N_PATCHES * PATCH_DIM * 4 / 1024:.1f} KB")
print(f"Memory/50 songs    : {50 * N_PATCHES * PATCH_DIM * 4 / 1024:.1f} KB  ← tiny")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 4.2 — SINGLE IMAGE WALKTHROUGH
# ─────────────────────────────────────────────────────────────
# Full pipeline on one image, shape printed at every stage.

df = pd.read_csv(SUBSET_CSV)
row = df.iloc[0]
song_id = str(row['song_id'])

# Find image file
candidate_paths = [
    IMAGE_DIR / f"{song_id}.jpg",
    IMAGE_DIR / f"{song_id}.jpeg",
    IMAGE_DIR / f"{song_id}.png",
    Path(row.get('image_path', 'MISSING')),
]
img_path = next((p for p in candidate_paths if p.exists()), None)

if img_path is None:
    print("⚠️  Image file not found. Generating synthetic RGB image for demonstration.")
    # Create a synthetic 256×300 random image (irregular size to test resizing)
    rng = np.random.default_rng(42)
    synthetic_arr = rng.integers(0, 256, (300, 256, 3), dtype=np.uint8)
    pil_img = Image.fromarray(synthetic_arr)
else:
    print(f"✅ Loading: {img_path}")
    pil_img = Image.open(img_path)

print()
print("STAGE 1: PIL Image")
print(f"  Mode         : {pil_img.mode}  (target: RGB)")
print(f"  Size (W, H)  : {pil_img.size}")

# Convert to RGB (handles RGBA, grayscale, CMYK, etc.)
pil_rgb = pil_img.convert('RGB')
print(f"  After RGB conv : {pil_rgb.mode}")
print()

print("STAGE 2: Resize to 224×224")
pil_resized = pil_rgb.resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS)
print(f"  PIL size       : {pil_resized.size}  (W, H)")
print()

print("STAGE 3: Convert to NumPy array")
img_arr = np.array(pil_resized, dtype=np.float32)  # (224, 224, 3) values in [0, 255]
print(f"  numpy shape    : {img_arr.shape}  (H, W, C)")
print(f"  dtype          : {img_arr.dtype}")
print(f"  value range    : [{img_arr.min():.0f}, {img_arr.max():.0f}]")
print()

print("STAGE 4: Scale to [0, 1]")
img_01 = img_arr / 255.0
print(f"  value range    : [{img_01.min():.3f}, {img_01.max():.3f}]")
print()

print("STAGE 5: ImageNet Normalization (per channel)")
img_norm = (img_01 - IMG_MEAN) / IMG_STD  # Broadcasting: (224, 224, 3) - (3,)
print(f"  Mean after norm (R,G,B): [{img_norm[:,:,0].mean():.3f}, "
      f"{img_norm[:,:,1].mean():.3f}, {img_norm[:,:,2].mean():.3f}]")
print(f"  Std after norm  (R,G,B): [{img_norm[:,:,0].std():.3f}, "
      f"{img_norm[:,:,1].std():.3f}, {img_norm[:,:,2].std():.3f}]")
print(f"  Shape           : {img_norm.shape}  (H, W, C)")
print()

print("STAGE 6: Channels-first format (C, H, W)")
# PyTorch convention: (C, H, W) instead of numpy/PIL convention (H, W, C)
img_chw = np.transpose(img_norm, (2, 0, 1)).astype(np.float32)
print(f"  Shape (C, H, W) : {img_chw.shape}")
print()

assert img_chw.shape == (N_CHANNELS, IMG_SIZE, IMG_SIZE), f"Shape mismatch: {img_chw.shape}"
assert not np.any(np.isnan(img_chw)), "NaN in image tensor!"
print("✅ Image tensor verified.")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 4.3 — MANUAL PATCH EXTRACTION (FROM SCRATCH)
# ─────────────────────────────────────────────────────────────
# No einops, no unfold — pure numpy nested loops (clarity > speed)

def extract_image_patches(img_chw: np.ndarray, patch_size: int) -> np.ndarray:
    """
    Extract non-overlapping patches from an image tensor.
    
    Args:
        img_chw  : np.ndarray shape (C, H, W) — channels-first image
        patch_size: int — size of each square patch (P)
    
    Returns:
        patches: np.ndarray shape (N_patches, C*P*P)
                 where N_patches = (H//P) * (W//P)
    
    Visualization:
        224×224 image with patch_size=16:
        ┌─────────────────────────────────────┐
        │ p00 p01 p02 ... p0,13              │
        │ p10 p11 ...                         │
        │ ...                                 │
        │ p13,0 ... p13,13                   │
        └─────────────────────────────────────┘
        Each cell = 16×16 = 256 pixels × 3 channels = 768 values
        Total: 14×14 = 196 patches
    """
    C, H, W = img_chw.shape
    assert H % patch_size == 0, f"H={H} not divisible by patch_size={patch_size}"
    assert W % patch_size == 0, f"W={W} not divisible by patch_size={patch_size}"
    
    n_rows = H // patch_size  # 14
    n_cols = W // patch_size  # 14
    patch_dim = C * patch_size * patch_size  # 768
    
    patches = []
    for r in range(n_rows):
        for c in range(n_cols):
            # Extract patch: all channels, row slice, col slice
            patch = img_chw[:,
                             r * patch_size : (r+1) * patch_size,
                             c * patch_size : (c+1) * patch_size]
            # patch shape: (C, patch_size, patch_size) = (3, 16, 16)
            # Flatten: C-first order → (C*P*P,) = (768,)
            patches.append(patch.flatten())
    
    return np.array(patches, dtype=np.float32)  # (N_patches, patch_dim)


# ── Apply ─────────────────────────────────────────────────────
print("Patch Extraction:")
print(f"  Input image  : {img_chw.shape}  (C, H, W)")

patches = extract_image_patches(img_chw, PATCH_SIZE)

print(f"  n_rows = H // P = {IMG_SIZE} // {PATCH_SIZE} = {N_PATCHES_PER_DIM}")
print(f"  n_cols = W // P = {IMG_SIZE} // {PATCH_SIZE} = {N_PATCHES_PER_DIM}")
print(f"  N_patches      = {N_PATCHES_PER_DIM} × {N_PATCHES_PER_DIM} = {N_PATCHES}")
print(f"  Patch dim      = {N_CHANNELS} × {PATCH_SIZE} × {PATCH_SIZE} = {PATCH_DIM}")
print(f"  Output shape   : {patches.shape}  ← (N_patches, patch_dim)")
print()

# Sanity: reconstruct image from patches
def reconstruct_image_from_patches(patches, C, H, W, patch_size):
    """Inverse of extract_image_patches — verify no information loss."""
    recon = np.zeros((C, H, W), dtype=np.float32)
    n_rows = H // patch_size
    n_cols = W // patch_size
    for idx, (r, c) in enumerate((r, c) for r in range(n_rows) for c in range(n_cols)):
        patch = patches[idx].reshape(C, patch_size, patch_size)
        recon[:, r*patch_size:(r+1)*patch_size, c*patch_size:(c+1)*patch_size] = patch
    return recon

recon_chw = reconstruct_image_from_patches(patches, N_CHANNELS, IMG_SIZE, IMG_SIZE, PATCH_SIZE)
max_err = np.max(np.abs(recon_chw - img_chw))
print(f"  Reconstruction max error: {max_err:.2e}  (should be ~0)")
assert max_err < 1e-5, f"❌ Reconstruction error: {max_err}"
print("✅ Patch extraction is lossless.")
print()
print(f"  Transformer will see: sequence of {N_PATCHES} tokens, each dim {PATCH_DIM}")
print(f"  After Linear({PATCH_DIM} → 128): ({N_PATCHES}, 128) per image")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 4.4 — VISUALIZATION: ORIGINAL, RESIZED, PATCHES
# ─────────────────────────────────────────────────────────────

fig = plt.figure(figsize=(18, 8))
gs  = gridspec.GridSpec(2, 8, figure=fig, hspace=0.3, wspace=0.2)

# 1. Original image
ax0 = fig.add_subplot(gs[:, :2])
# Convert normalized (C,H,W) back to displayable (H,W,3) uint8
display_img = np.transpose(img_chw, (1, 2, 0))  # (H, W, C)
display_img = (display_img * IMG_STD + IMG_MEAN)  # denormalize
display_img = np.clip(display_img, 0, 1)
ax0.imshow(display_img)
ax0.set_title(f'Processed Image\n{IMG_SIZE}×{IMG_SIZE} RGB', fontsize=10)
ax0.axis('off')

# Draw patch grid overlay
for r in range(N_PATCHES_PER_DIM + 1):
    ax0.axhline(r * PATCH_SIZE - 0.5, color='yellow', linewidth=0.5, alpha=0.7)
for c in range(N_PATCHES_PER_DIM + 1):
    ax0.axvline(c * PATCH_SIZE - 0.5, color='yellow', linewidth=0.5, alpha=0.7)

# 2. Show first 12 patches individually
for idx in range(min(12, N_PATCHES)):
    row_pos = idx // 6
    col_pos = idx % 6
    ax = fig.add_subplot(gs[row_pos, 2 + col_pos])
    patch_rgb = patches[idx].reshape(N_CHANNELS, PATCH_SIZE, PATCH_SIZE)
    patch_display = np.transpose(patch_rgb, (1, 2, 0))
    patch_display = patch_display * IMG_STD + IMG_MEAN
    patch_display = np.clip(patch_display, 0, 1)
    ax.imshow(patch_display)
    ax.set_title(f'P{idx}', fontsize=7)
    ax.axis('off')

fig.suptitle(f'Image Patching Pipeline: {N_PATCHES} patches of {PATCH_SIZE}×{PATCH_SIZE}×3={PATCH_DIM} values each',
             fontsize=11, fontweight='bold')
plt.savefig(FIGURES_DIR / 'fig_patches.png', dpi=100, bbox_inches='tight')
plt.show()

# PCA of patches — reveal spatial structure
from numpy.linalg import svd
# Center patches
p_centered = patches - patches.mean(axis=0)
# SVD for PCA
U, S, Vt = svd(p_centered, full_matrices=False)
patches_2d = U[:, :2] * S[:2]  # project to 2D

fig2, ax2 = plt.subplots(figsize=(7, 6))
scatter = ax2.scatter(patches_2d[:, 0], patches_2d[:, 1],
                      c=range(N_PATCHES), cmap='plasma', s=50, alpha=0.8)
plt.colorbar(scatter, ax=ax2, label='Patch index (raster order)')
ax2.set_title('PCA of Image Patches (2D projection)\nSpatially adjacent patches cluster together')
ax2.set_xlabel('PC1')
ax2.set_ylabel('PC2')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_patch_pca.png', dpi=120)
plt.show()
print("✅ Figures saved.")
print("PCA intuition: patches from similar regions (sky, background, face)")
print("cluster in the same region of the 2D projection space.")
print("The Vision Transformer learns to make use of this structure via self-attention.")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 4.5 — FULL PIPELINE FUNCTION + BATCH PROCESSING
# ─────────────────────────────────────────────────────────────

def preprocess_image(image_path: str,
                     img_size: int = IMG_SIZE,
                     patch_size: int = PATCH_SIZE,
                     mean: np.ndarray = IMG_MEAN,
                     std: np.ndarray = IMG_STD) -> np.ndarray:
    """
    Full image preprocessing pipeline.
    Returns patches: np.ndarray shape (N_patches, C*P*P), dtype float32.
    Returns None on failure.
    """
    try:
        pil_img = Image.open(image_path).convert('RGB')
        pil_img = pil_img.resize((img_size, img_size), Image.LANCZOS)
        arr     = np.array(pil_img, dtype=np.float32) / 255.0  # (H, W, 3) in [0,1]
        arr     = (arr - mean) / std                             # normalize
        arr     = np.transpose(arr, (2, 0, 1)).astype(np.float32)  # (3, H, W)
        patches = extract_image_patches(arr, patch_size)
        return patches
    except Exception as e:
        print(f"  ⚠️  Error: {image_path} → {e}")
        return None


print("Batch processing all images...")

results = []
for _, row in tqdm(df.iterrows(), total=len(df), desc="Image preprocessing"):
    song_id = str(row['song_id'])
    out_path = PROC_IMAGE / f"{song_id}.npy"
    
    # Find image
    img_candidates = [
        IMAGE_DIR / f"{song_id}.jpg",
        IMAGE_DIR / f"{song_id}.jpeg",
        IMAGE_DIR / f"{song_id}.png",
        Path(row.get('image_path', 'MISSING')),
    ]
    found = next((p for p in img_candidates if p.exists()), None)
    
    if found is None:
        results.append({'song_id': song_id, 'status': 'MISSING'})
        continue
    
    if out_path.exists():
        results.append({'song_id': song_id, 'status': 'CACHED'})
        continue
    
    patches = preprocess_image(str(found))
    if patches is not None:
        np.save(out_path, patches)
        results.append({'song_id': song_id, 'status': 'OK', 'shape': patches.shape})
    else:
        results.append({'song_id': song_id, 'status': 'ERROR'})

df_res = pd.DataFrame(results)
print()
print("Summary:", df_res['status'].value_counts().to_dict())

# Verification
saved = sorted(PROC_IMAGE.glob('*.npy'))
issues = []
for fp in saved:
    arr = np.load(fp)
    if arr.shape != (N_PATCHES, PATCH_DIM):
        issues.append(f"  ❌ {fp.stem}: wrong shape {arr.shape}")
    elif np.any(np.isnan(arr)):
        issues.append(f"  ❌ {fp.stem}: NaN")

if not issues:
    print(f"✅ All {len(saved)} image files verified: shape=({N_PATCHES}, {PATCH_DIM}), no NaN")
else:
    for i in issues:
        print(i)

---
## ✅ Notebook 4 Complete

**What was built:**
- Mathematical understanding of ViT patching and ImageNet normalization
- Full image pipeline: load → RGB → resize 224×224 → normalize → (C,H,W) → patches
- From-scratch patch extraction with lossless reconstruction verification
- PCA visualization of patch structure
- Batch processing for all 50 songs → `(196, 768)` .npy files

**In the Vision Transformer:**
```python
# patches  : (B, 196, 768)
# + CLS token prepended: (B, 197, 768)
# After Linear(768 → 128): (B, 197, 128)
# After Transformer: (B, 197, 128)
# output[:, 0, :] → (B, 128)  ← image embedding (CLS token)
```

**Next:** Notebook 5 — All Three Transformers From Scratch (Audio + Lyrics + Vision)